# Yield Curve Forecasting

Nelson-Siegel baseline fit, RUB–CNY cross-currency spreads, and evaluation.

**Run the data pipeline first** to populate the database.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.forecasting.loaders import load_russian_yield_curve, load_chinese_yield_curve
from src.forecasting.ns_baseline import compute_residuals, get_fitted_curves
from src.forecasting.cross_currency import build_spreads, flag_abnormal_spreads
from src.forecasting.evaluate import evaluate_ns_fit, evaluate_spreads

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100
print('Setup complete.')

## 1. Load Yield Curves

In [ ]:
# Optional: restrict date range
START_DATE = None  # e.g. '2019-01-01'
END_DATE = None   # e.g. '2025-12-31'

ru_yields = load_russian_yield_curve(start_date=START_DATE, end_date=END_DATE)
cn_yields = load_chinese_yield_curve(start_date=START_DATE, end_date=END_DATE)
ru_residuals = pd.DataFrame()  # filled by NS fit cell
cn_residuals = pd.DataFrame()

print(f'RU: {len(ru_yields)} rows, {list(ru_yields.columns)}')
print(f'CN: {len(cn_yields)} rows, {list(cn_yields.columns)}')

if ru_yields.empty and cn_yields.empty:
    print('No yield data. Run the data pipeline first.')

## 2. Nelson-Siegel Baseline Fit

In [ ]:
if not ru_yields.empty:
    ru_fitted = get_fitted_curves(ru_yields)
    ru_residuals = compute_residuals(ru_yields)
    ru_eval = evaluate_ns_fit(ru_yields, ru_fitted, ru_residuals)
    print('RU Nelson-Siegel:')
    print(f"  MAE mean: {ru_eval.get('overall', {}).get('MAE_mean', 'N/A')}")
    print(f"  RMSE mean: {ru_eval.get('overall', {}).get('RMSE_mean', 'N/A')}")
    display(pd.DataFrame(ru_eval.get('by_maturity', {})).T)
else:
    print('No RU data.')

In [ ]:
if not cn_yields.empty:
    cn_fitted = get_fitted_curves(cn_yields)
    cn_residuals = compute_residuals(cn_yields)
    cn_eval = evaluate_ns_fit(cn_yields, cn_fitted, cn_residuals)
    print('CN Nelson-Siegel:')
    print(f"  MAE mean: {cn_eval.get('overall', {}).get('MAE_mean', 'N/A')}")
    print(f"  RMSE mean: {cn_eval.get('overall', {}).get('RMSE_mean', 'N/A')}")
    display(pd.DataFrame(cn_eval.get('by_maturity', {})).T)
else:
    print('No CN data.')

## 3. Cross-Currency Spreads (RUB–CNY)

In [ ]:
spreads = build_spreads(ru_yields, cn_yields)
if spreads.empty:
    print('No common maturities or overlapping dates.')
else:
    flagged = flag_abnormal_spreads(spreads, z_threshold=2.0)
    spread_eval = evaluate_spreads(spreads, flagged)
    print('Spread stats:')
    display(pd.DataFrame(spread_eval.get('spread_stats', {})).T)
    print('Abnormal counts (Z>2):')
    display(pd.Series(spread_eval.get('abnormal_counts', {})))
    spreads.plot(title='RUB–CNY yield spreads by maturity')
    plt.tight_layout()
    plt.show()

## 4. Residual Plots (RU)

In [ ]:
if not ru_yields.empty and not ru_residuals.empty:
    ru_residuals.plot(title='RU Nelson-Siegel residuals by maturity')
    plt.tight_layout()
    plt.show()
else:
    print('No RU residuals.')